## tl;dr

Массовые бесплатные дни уже дважды совпали с резким провалом оплат. Рекомендация: не давать месяц за одного друга и не давать дополнительные дни приглашённому. Запустить ограниченный 30-дневный тест **«2 новых оплативших друга → 30 дней рефереру»** поверх текущих 20%, только для новых событий и с контрольной группой.

## Context & Methods

### Key Assumptions

- Источник: read-only production SQLite `/opt/doodlevpn-data/bot.db`, срез 31 июля 2026 года.
- Успешная оплата дедуплицирована по provider payment row; Stars берутся из канонического ledger.
- Майская раздача восстановлена из `mass_grant_runs/results`; июльская — из `broadcasts/broadcast_jobs` и платёжной временной серии.
- Это естественные эксперименты без holdout: техработы, сезонность и июльская скидка 37% мешают причинной оценке. Поэтому цифры используются как риск-сигнал, не как точный causal lift.

In [1]:
gift_windows = [{"event": "+3 дня, май", "phase": "До", "phase_order": 1, "days": 3, "recipients": 5179, "payments": 41, "first": 16, "repeat": 25}, {"event": "+3 дня, май", "phase": "Подарок", "phase_order": 2, "days": 3, "recipients": 5179, "payments": 27, "first": 13, "repeat": 14}, {"event": "+3 дня, май", "phase": "После", "phase_order": 3, "days": 3, "recipients": 5179, "payments": 48, "first": 32, "repeat": 16}, {"event": "+7 дней, июль", "phase": "До", "phase_order": 1, "days": 7, "recipients": 4707, "payments": 147, "first": 29, "repeat": 118}, {"event": "+7 дней, июль", "phase": "Подарок", "phase_order": 2, "days": 7, "recipients": 4707, "payments": 56, "first": 19, "repeat": 37}, {"event": "+7 дней, июль", "phase": "После", "phase_order": 3, "days": 7, "recipients": 4707, "payments": 150, "first": 47, "repeat": 103}]
for event in sorted(set(r['event'] for r in gift_windows)):
    rows = sorted((r for r in gift_windows if r['event'] == event), key=lambda r: r['phase_order'])
    before, gift, after = rows
    print(event, 'payment change during gift:', round((gift['payments']/before['payments']-1)*100, 1), '%', 'repeat change:', round((gift['repeat']/before['repeat']-1)*100, 1), '%')

+3 дня, май payment change during gift: -34.1 % repeat change: -44.0 %
+7 дней, июль payment change during gift: -61.9 % repeat change: -68.6 %


## Data

Равные по длине окна сравниваются вокруг момента начисления. В июле окно после подарка содержит 44 оплаты со скидкой 37%, поэтому восстановление числа транзакций завышает восстановление выручки.

In [2]:
monthly_thresholds = [{"month": "Апрель с 13-го", "paid_friends": 66, "active_referrers": 59, "reward_1": 59, "reward_2": 6, "reward_3": 1}, {"month": "Май", "paid_friends": 94, "active_referrers": 76, "reward_1": 76, "reward_2": 12, "reward_3": 4}, {"month": "Июнь", "paid_friends": 105, "active_referrers": 85, "reward_1": 85, "reward_2": 15, "reward_3": 4}, {"month": "Июль", "paid_friends": 57, "active_referrers": 51, "reward_1": 51, "reward_2": 6, "reward_3": 0}]
for r in monthly_thresholds:
    print(r['month'], 'paid friends=', r['paid_friends'], 'reward users at thresholds 1/2/3=', r['reward_1'], r['reward_2'], r['reward_3'])

Апрель с 13-го paid friends= 66 reward users at thresholds 1/2/3= 59 6 1
Май paid friends= 94 reward users at thresholds 1/2/3= 76 12 4
Июнь paid friends= 105 reward users at thresholds 1/2/3= 85 15 4
Июль paid friends= 57 reward users at thresholds 1/2/3= 51 6 0


## Results

В июле за точные семь дней до раздачи было 147 оплат (118 повторных), во время подарка — 56 (37 повторных), после — 150 (103 повторных). За первые 23,1 часа после окончания подарка, ещё до массовой рассылки скидки, прошло 30 оплат против 56 за все 168 часов подарочного окна.

В мае эффект слабее, но направлен так же: повторные оплаты 25 → 14 в трёхдневное окно (−44%).

In [3]:
variants = [{"variant": "30 дней за 1 оплатившего друга", "july_reward_users": 51, "service_months": 51.0, "push": "очень сильный", "cashflow_risk": "очень высокий", "decision": "не запускать"}, {"variant": "30 дней за 2 оплативших друзей", "july_reward_users": 6, "service_months": 6.0, "push": "сильный", "cashflow_risk": "низкий и ограничиваемый", "decision": "рекомендация"}, {"variant": "30 дней за 3 оплативших друзей", "july_reward_users": 0, "service_months": 0.0, "push": "слабый из-за недостижимости", "cashflow_risk": "низкий", "decision": "не запускать первым"}, {"variant": "14 дней за каждого оплатившего", "july_reward_users": 51, "service_months": 26.6, "push": "средний", "cashflow_risk": "средний", "decision": "резерв"}, {"variant": "7 дней за каждого оплатившего", "july_reward_users": 51, "service_months": 13.3, "push": "слабый-средний", "cashflow_risk": "средний", "decision": "проигрывает порогу 2"}, {"variant": "+7 дней приглашённому после оплаты", "july_reward_users": 57, "service_months": 13.3, "push": "средний", "cashflow_risk": "прямо откладывает его продление", "decision": "не запускать"}, {"variant": "Только текущие 20%", "july_reward_users": 0, "service_months": 0.0, "push": "слабый", "cashflow_risk": "низкий", "decision": "оставить как основу, но не как толчок"}, {"variant": "Лотерея / кейсы", "july_reward_users": 0, "service_months": 0.0, "push": "неизвестный", "cashflow_risk": "непредсказуемый + fraud", "decision": "не сейчас"}]
for r in variants:
    print(f"{r['variant']} | July service-months={r['service_months']} | {r['decision']}")

30 дней за 1 оплатившего друга | July service-months=51.0 | не запускать
30 дней за 2 оплативших друзей | July service-months=6.0 | рекомендация
30 дней за 3 оплативших друзей | July service-months=0.0 | не запускать первым
14 дней за каждого оплатившего | July service-months=26.6 | резерв
7 дней за каждого оплатившего | July service-months=13.3 | проигрывает порогу 2
+7 дней приглашённому после оплаты | July service-months=13.3 | не запускать
Только текущие 20% | July service-months=0.0 | оставить как основу, но не как толчок
Лотерея / кейсы | July service-months=0.0 | не сейчас


## Takeaways

- Месяц за одного друга охватил бы 51 июльского реферера и создал бы 51 бесплатный пользователь-месяц на базовом объёме — до доказанного прироста.
- Порог три в июле не достиг никто; такой оффер почти не создаёт обратной связи.
- Порог два дал бы только 6 бесплатных месяцев на базовом июльском потоке, но сохраняет сильный заголовок и понятный прогресс 0/2 → 1/2 → 2/2.
- Тест должен считать только первые оплаты прямых друзей после старта, исключать специальных партнёров и выдавать награду после 48-часового hold.
- Масштабирование допустимо только по incremental paid referrals и incremental contribution относительно контроля.